# Análisis XAI del modelo de segmentación cerebral

Este cuaderno acompaña la memoria del TFG **Herramienta de análisis de imágenes biomédicas**, desarrollado en colaboración con **ICTS BioImagen Complutense**. Su finalidad es documentar y ejecutar un análisis de explicabilidad del modelo de segmentación binaria del cerebro en resonancia magnética preclínica de ratón.

El objetivo no es sustituir la evaluación cuantitativa de segmentación, sino estudiar si las regiones destacadas por distintos métodos de XAI son espacialmente coherentes con la predicción anatómica del modelo. Las explicaciones se interpretan como una herramienta de inspección y depuración, no como una prueba causal ni como validación clínica independiente.

Métodos incluidos: Grad-CAM, Grad-CAM++, Integrated Gradients, sensibilidad por oclusión, curvas de Deletion/Insertion, curvas MoRF/LeRF y AOPC.

Flujo general del análisis:

1. Configurar rutas, umbrales y parámetros de XAI.
2. Cargar funciones auxiliares de preprocesado, inferencia, visualización y métricas.
3. Resolver la capa convolucional usada por los métodos tipo CAM.
4. Ejecutar pruebas rápidas sobre cortes representativos.
5. Procesar todos los cortes de todos los volúmenes `CASE_*`.
6. Exportar métricas por corte, por método y por punto de curva.
7. Seleccionar ejemplos cualitativos y guardar montajes con imagen original, máscara real, predicción y mapas XAI.

Decisiones metodológicas principales:

- El modelo de cerebro recibe directamente el corte de RM, sin aplicar una máscara cerebral previa.
- La salida analizada es la probabilidad de primer plano de una tarea de segmentación binaria, no una puntuación de clasificación.
- El objetivo escalar de XAI se define como la probabilidad media dentro de la máscara cerebral predicha por el propio modelo.
- Las métricas de fidelidad solo se agregan para cortes con una predicción cerebral no vacía.
- Los resultados se guardan en una carpeta con marca temporal dentro del directorio del conjunto de datos.


## 1. Configuración del experimento

Esta celda reúne los parámetros que normalmente debe revisar la persona que ejecute el cuaderno: rutas del conjunto de datos y del modelo, umbral de segmentación, capa objetivo para XAI y resolución de las curvas de perturbación.

Para mantener la reproducibilidad, las salidas se escriben en una carpeta nueva identificada por fecha y hora. Si se cambia la modalidad de RM, el modelo o el preprocesado, conviene dejarlo anotado en esta sección antes de interpretar los resultados.


In [ ]:
from pathlib import Path
from datetime import datetime

DATASET_DIR = "test_dataset"
MODEL_BRAIN_PATH = "models/brain_unet_model.h5"

IMG_TARGET = 120
BRAIN_THRESHOLD = 0.5

TARGET_LAYER_NAME = None
SCORE_MODE = "pred_mask_mean_probability"
BRAIN_MODEL_OUTPUTS_LOGITS = None  # None = infer from the final activation; True = apply sigmoid once; False = already probabilities.
PERTURBATION_BASELINE = "mean"  # "mean" is the default for quantitative curves; "black" can be tested for sensitivity.

IG_STEPS = 64
CURVE_STEPS = 20
OCCLUSION_PATCH = 12
OCCLUSION_STRIDE = 6
SELECTED_SLICES_PER_GROUP = 6

ALLOW_TOPK_FALLBACK_FOR_VISUALIZATION = False
TOPK_FALLBACK_FRACTION = 0.01
TOPK_FALLBACK_MIN_PIXELS = 32
LOW_BRAIN_AREA_FRACTION_QUANTILE = 0.15
SMALL_PRED_BRAIN_AREA_FRACTION_QUANTILE = 0.15

# Memory-oriented defaults
SCORING_BATCH_SIZE = 8
GRADCAMPP_MEMORY_SAFE = True
SMOKE_TEST_IG_STEPS = 16
SMOKE_TEST_CURVE_STEPS = 6
SMOKE_TEST_OCCLUSION_PATCH = 24
SMOKE_TEST_OCCLUSION_STRIDE = 24
SMOKE_TEST_CURVE_METHODS = ("gradcam",)

OPTIONAL_INSTALL_MISSING_DEPENDENCIES = False

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_ROOT = Path(DATASET_DIR) / "_xai_cerebro_analysis" / timestamp
CSV_DIR = OUTPUT_ROOT / "csv"
FIGURES_DIR = OUTPUT_ROOT / "figures"

CSV_DIR.mkdir(parents=True, exist_ok=True)
for group_name in (
    "good_brain_segmentation",
    "medium_brain_segmentation",
    "low_brain_segmentation",
    "low_gt_brain_area",
    "low_pred_brain_area",
    "interior",
    "edge_first_last3",
):
    (FIGURES_DIR / group_name).mkdir(parents=True, exist_ok=True)

print("Configuration ready")
print(f"DATASET_DIR: {DATASET_DIR}")
print(f"MODEL_BRAIN_PATH: {MODEL_BRAIN_PATH}")
print(f"OUTPUT_ROOT: {OUTPUT_ROOT}")
print(f"SCORING_BATCH_SIZE: {SCORING_BATCH_SIZE}")
print(f"GRADCAMPP_MEMORY_SAFE: {GRADCAMPP_MEMORY_SAFE}")
print(f"SCORE_MODE: {SCORE_MODE}")
print(f"PERTURBATION_BASELINE: {PERTURBATION_BASELINE}")


## 2. Comprobación del entorno

Antes de cargar los modelos se verifica que las dependencias principales estén disponibles. Esta comprobación facilita la ejecución en entornos locales o en notebooks en la nube, y deja constancia de las versiones usadas durante el análisis.


In [ ]:
import importlib
import platform
import subprocess
import sys

REQUIRED_PACKAGES = {
    "tensorflow": "tensorflow",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "PIL": "Pillow",
    "nibabel": "nibabel",
}

missing_packages = []
imported_versions = {}

for module_name, pip_name in REQUIRED_PACKAGES.items():
    try:
        module = importlib.import_module(module_name)
        imported_versions[module_name] = getattr(module, "__version__", "unknown")
    except Exception:
        missing_packages.append((module_name, pip_name))

print(f"Python: {platform.python_version()}")
if imported_versions:
    print("Imported package versions:")
    for module_name, version in imported_versions.items():
        print(f"  - {module_name}: {version}")
else:
    print("No required packages are currently importable in this environment.")

if missing_packages:
    print("Missing packages detected:")
    for module_name, pip_name in missing_packages:
        print(f"  - {module_name} (pip install {pip_name})")
else:
    print("All required packages are available.")

if missing_packages and OPTIONAL_INSTALL_MISSING_DEPENDENCIES:
    install_targets = [pip_name for _, pip_name in missing_packages]
    print(f"Installing missing packages: {install_targets}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install_targets])
    print("Installation complete. Re-run this cell, then continue.")
elif missing_packages:
    print("Set OPTIONAL_INSTALL_MISSING_DEPENDENCIES = True and re-run this cell to install automatically, or install manually.")


## 3. Importación de librerías

A partir de este punto el cuaderno prepara las librerías científicas, de visualización y de aprendizaje profundo necesarias para procesar volúmenes NIfTI, máscaras 2D y modelos TensorFlow/Keras.


In [ ]:
if missing_packages:
    missing_names = ", ".join(module_name for module_name, _ in missing_packages)
    raise ImportError(
        "Cannot continue because the following packages are missing: "
        f"{missing_names}. Install them first or enable OPTIONAL_INSTALL_MISSING_DEPENDENCIES."
    )

import gc
import math
import os
import re
import warnings
from collections import defaultdict

import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from PIL import Image, ImageFilter
from tensorflow.keras import Model
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.models import load_model

np.random.seed(42)
tf.random.set_seed(42)
warnings.filterwarnings("ignore", category=UserWarning)

plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["image.cmap"] = "gray"

print(f"TensorFlow version: {tf.__version__}")


## 4. Funciones auxiliares de segmentación y preprocesado

Este bloque define utilidades comunes: pérdidas usadas al cargar el modelo, lectura de imágenes y máscaras, normalización, redimensionado explícito a `120 x 120` e inferencia del modelo cerebral.

Las máscaras correspondientes a un mismo corte se combinan mediante unión cuando existen varias anotaciones. Esto evita tratar regiones separadas como ejemplos independientes y mantiene la correspondencia corte-máscara de forma transparente.


In [ ]:
# ============================================================
# Helper functions: losses, I/O, preprocessing, inference
# ============================================================

def dice_coef(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + dice_loss(y_true, y_pred)

def dice_numpy(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.uint8)
    y_pred = y_pred.astype(np.uint8)
    total = y_true.sum() + y_pred.sum()
    if total == 0:
        return 1.0
    intersection = np.sum(y_true * y_pred)
    return (2.0 * intersection + smooth) / (total + smooth)

def iou_numpy(y_true, y_pred, smooth=1e-6):
    y_true = y_true.astype(np.uint8)
    y_pred = y_pred.astype(np.uint8)
    union = np.sum((y_true + y_pred) > 0)
    if union == 0:
        return 1.0
    intersection = np.sum(y_true * y_pred)
    return (intersection + smooth) / (union + smooth)

def normalize_map(array_2d):
    arr = np.asarray(array_2d, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    arr_min = float(arr.min())
    arr_max = float(arr.max())
    if arr_max - arr_min < 1e-8:
        return np.zeros_like(arr, dtype=np.float32)
    return (arr - arr_min) / (arr_max - arr_min)

def infer_model_outputs_logits(model, configured_value=None):
    """Return True only when the model output should be converted with sigmoid.

    The current project models usually end with a sigmoid activation, so their
    outputs are already probabilities. This guard prevents accidentally applying
    sigmoid twice, while still supporting future models that output logits.
    """
    if configured_value is not None:
        return bool(configured_value)
    activation = getattr(model.layers[-1], "activation", None)
    activation_name = getattr(activation, "__name__", "")
    return activation_name in {"linear", "identity"}

def get_resolved_brain_outputs_logits(outputs_logits=None):
    """Return the resolved logits/probability flag used by score functions."""
    resolved_value = BRAIN_MODEL_OUTPUTS_LOGITS if outputs_logits is None else outputs_logits
    if resolved_value is None:
        raise RuntimeError(
            "BRAIN_MODEL_OUTPUTS_LOGITS has not been resolved yet. "
            "Load the brain model with load_brain_model() before scoring."
        )
    return bool(resolved_value)

def output_to_probability_tf(model_output_tf, outputs_logits=None):
    values = tf.cast(model_output_tf, tf.float32)
    if get_resolved_brain_outputs_logits(outputs_logits):
        values = tf.math.sigmoid(values)
    return tf.clip_by_value(values, 0.0, 1.0)

def output_to_probability_np(model_output_np, outputs_logits=None):
    values = np.asarray(model_output_np, dtype=np.float32)
    if get_resolved_brain_outputs_logits(outputs_logits):
        values = 1.0 / (1.0 + np.exp(-values))
    return np.clip(values, 0.0, 1.0).astype(np.float32)

def first_model_output(model_output):
    """Return the first tensor when a Keras model returns a list/tuple output."""
    if isinstance(model_output, (list, tuple)):
        if not model_output:
            raise ValueError("Model returned an empty output list.")
        return model_output[0]
    return model_output

def foreground_channel(model_output):
    """Extract the single foreground channel from a model output tensor."""
    output = first_model_output(model_output)
    return output[..., 0]

def parse_slice_idx_from_path(path_obj):
    stem = Path(path_obj).stem
    prefix = stem.split("-")[0]
    if prefix.isdigit():
        return int(prefix) - 1
    numbers = re.findall(r"\d+", stem)
    if not numbers:
        raise ValueError(f"Could not parse slice index from: {path_obj}")
    return int(numbers[0]) - 1

def load_binary_mask(mask_path):
    image = Image.open(mask_path).convert("L")
    mask = np.asarray(image, dtype=np.uint8)
    return (mask > 127).astype(np.uint8)

def load_union_mask_dict(mask_dir, vol_shape=None):
    mask_dir = Path(mask_dir)
    union_masks = {}
    if not mask_dir.exists():
        return union_masks
    for png_path in sorted(mask_dir.glob("*.png")):
        slice_idx = parse_slice_idx_from_path(png_path)
        if vol_shape is not None and (slice_idx < 0 or slice_idx >= vol_shape[2]):
            continue
        current_mask = load_binary_mask(png_path)
        if slice_idx in union_masks:
            union_masks[slice_idx] = np.maximum(union_masks[slice_idx], current_mask)
        else:
            union_masks[slice_idx] = current_mask
    return union_masks

def load_nifti_volume(nifti_path):
    nii = nib.load(str(nifti_path))
    volume = nii.get_fdata()
    if volume.ndim == 4:
        volume = np.squeeze(volume)
    if volume.ndim != 3:
        raise ValueError(f"Expected a 3D volume after squeeze, got shape {volume.shape} for {nifti_path}")
    volume = volume.astype(np.float32)
    volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
    volume = np.rot90(volume, k=1, axes=(0, 1))
    volume = np.flip(volume, axis=0)
    return nii, volume

def resize_image_slice(slice_img, target_size=IMG_TARGET):
    slice_img = np.asarray(slice_img, dtype=np.float32)
    if slice_img.shape != (target_size, target_size):
        image = Image.fromarray(np.clip(slice_img * 255.0, 0, 255).astype(np.uint8))
        image = image.resize((target_size, target_size))
        slice_img = np.asarray(image, dtype=np.float32) / 255.0
    return slice_img.astype(np.float32)

def resize_mask(mask, target_size=IMG_TARGET):
    mask = np.asarray(mask, dtype=np.uint8)
    if mask.shape != (target_size, target_size):
        image = Image.fromarray(mask)
        image = image.resize((target_size, target_size), resample=Image.NEAREST)
        mask = np.asarray(image, dtype=np.uint8)
    return (mask > 0).astype(np.uint8)

def build_target_roi(prob_map, threshold=BRAIN_THRESHOLD, allow_visual_fallback=ALLOW_TOPK_FALLBACK_FOR_VISUALIZATION, top_fraction=TOPK_FALLBACK_FRACTION, min_pixels=TOPK_FALLBACK_MIN_PIXELS):
    """Build the target ROI from the model prediction, not from ground truth.

    For quantitative XAI we explain the decision actually made by the model: the
    predicted brain mask. If the prediction is empty there is no foreground anatomical prediction to explain, so faithfulness metrics must be skipped instead of using
    a top-probability fallback that would invent a target.
    """
    prob_map = np.asarray(prob_map, dtype=np.float32)
    pred_mask = (prob_map > threshold).astype(np.uint8)
    if pred_mask.sum() > 0:
        return pred_mask, "predicted_mask", True

    if not allow_visual_fallback:
        return np.zeros_like(pred_mask, dtype=np.uint8), "no_predicted_brain", False

    total_pixels = prob_map.size
    top_k = max(int(math.ceil(total_pixels * float(top_fraction))), int(min_pixels))
    top_k = min(top_k, total_pixels)

    flat = prob_map.reshape(-1)
    ranked_idx = np.argsort(-flat)
    roi_flat = np.zeros_like(flat, dtype=np.uint8)
    roi_flat[ranked_idx[:top_k]] = 1
    return roi_flat.reshape(prob_map.shape), "top_probability_fallback_visual_only", False

def segmentation_score_from_model_output(model_output_tf, roi_mask_tf, eps=1e-8):
    """Mean foreground probability inside the predicted brain mask.

    This scalar is stable across different predicted brain-mask sizes because it averages
    probability inside the fixed ROI instead of summing over pixels.
    """
    prob_map_tf = output_to_probability_tf(model_output_tf)
    roi_mask_tf = tf.cast(roi_mask_tf, tf.float32)
    denominator = tf.reduce_sum(roi_mask_tf) + eps
    return tf.reduce_sum(prob_map_tf * roi_mask_tf) / denominator

def compute_scalar_score_numpy(model_output_np, roi_mask_np, eps=1e-8):
    roi = np.asarray(roi_mask_np, dtype=np.float32)
    if float(roi.sum()) <= 0.0:
        return np.nan
    prob = output_to_probability_np(model_output_np)
    return float(np.sum(prob * roi) / (np.sum(roi) + eps))

def make_blurred_baseline(image_2d, radius=3.0):
    image_uint8 = np.clip(np.asarray(image_2d, dtype=np.float32) * 255.0, 0, 255).astype(np.uint8)
    blurred = Image.fromarray(image_uint8).filter(ImageFilter.GaussianBlur(radius=radius))
    return np.asarray(blurred, dtype=np.float32) / 255.0

def make_perturbation_baseline(image_2d, mode=PERTURBATION_BASELINE):
    """Baseline used by Deletion/Insertion and occlusion perturbations.

    Both "black" and "mean" are supported. The default "mean" usually gives
    smoother MRI perturbation curves because it avoids adding hard black squares
    that can behave like artificial edges.
    """
    image = np.asarray(image_2d, dtype=np.float32)
    if mode == "black":
        return np.zeros_like(image, dtype=np.float32)
    if mode == "mean":
        mean_value = float(np.mean(image))
        return np.full_like(image, mean_value, dtype=np.float32)
    if mode == "blurred":
        return make_blurred_baseline(image)
    raise ValueError(f"Unsupported PERTURBATION_BASELINE={mode!r}. Use 'mean', 'black', or 'blurred'.")

def build_sliding_positions(size, patch, stride):
    positions = list(range(0, max(size - patch + 1, 1), stride))
    last_start = max(size - patch, 0)
    if positions[-1] != last_start:
        positions.append(last_start)
    return positions

def get_edge_position_label(slice_idx, total_slices):
    if slice_idx < min(3, total_slices):
        return f"first_{slice_idx + 1}"
    if slice_idx >= max(total_slices - 3, 0):
        return f"last_{total_slices - slice_idx}"
    return None

def sort_case_dirs(dataset_dir):
    dataset_path = Path(dataset_dir)
    case_dirs = [p for p in dataset_path.iterdir() if p.is_dir() and p.name.startswith("CASE_")]
    def case_sort_key(path_obj):
        parts = path_obj.name.split("_")
        if len(parts) > 1 and parts[1].isdigit():
            return int(parts[1])
        return path_obj.name
    return sorted(case_dirs, key=case_sort_key)

def resolve_target_layer_name(model, requested_name=None):
    if requested_name is not None:
        _ = model.get_layer(requested_name)
        return requested_name

    output_conv_idx = None
    for idx in range(len(model.layers) - 1, -1, -1):
        layer = model.layers[idx]
        if isinstance(layer, Conv2D) and getattr(layer, "filters", None) == 1 and tuple(layer.kernel_size) == (1, 1):
            output_conv_idx = idx
            break

    search_start = output_conv_idx - 1 if output_conv_idx is not None else len(model.layers) - 1
    for idx in range(search_start, -1, -1):
        layer = model.layers[idx]
        if isinstance(layer, Conv2D) and getattr(layer, "filters", 0) > 1:
            return layer.name

    raise ValueError("Could not resolve a decoder convolution layer before the final 1x1 output convolution.")

def load_brain_model(model_brain_path, target_layer_name=None):
    global BRAIN_MODEL_OUTPUTS_LOGITS
    custom_objects = {
        "dice_coef": dice_coef,
        "dice_loss": dice_loss,
        "bce_dice_loss": bce_dice_loss,
    }
    model_brain = load_model(model_brain_path, custom_objects=custom_objects)
    resolved_target_layer = resolve_target_layer_name(model_brain, requested_name=target_layer_name)
    detected_brain_outputs_logits = infer_model_outputs_logits(model_brain, BRAIN_MODEL_OUTPUTS_LOGITS)
    BRAIN_MODEL_OUTPUTS_LOGITS = bool(detected_brain_outputs_logits)
    return model_brain, resolved_target_layer, BRAIN_MODEL_OUTPUTS_LOGITS

def score_batch_inputs(model, batch_inputs, roi_mask, batch_size=SCORING_BATCH_SIZE):
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)
    if float(np.sum(roi_mask)) <= 0.0:
        return np.full((len(batch_inputs),), np.nan, dtype=np.float32)

    score_chunks = []
    total = len(batch_inputs)

    for start in range(0, total, max(int(batch_size), 1)):
        stop = min(start + max(int(batch_size), 1), total)
        inputs_tf = tf.convert_to_tensor(batch_inputs[start:stop], dtype=tf.float32)
        predictions = foreground_channel(model(inputs_tf, training=False))
        scores = []
        for batch_idx in range(predictions.shape[0]):
            scores.append(segmentation_score_from_model_output(predictions[batch_idx], roi_tf))
        scores = tf.stack(scores)
        score_chunks.append(scores.numpy().astype(np.float32))
        del inputs_tf
        del predictions
        del scores

    if not score_chunks:
        return np.asarray([], dtype=np.float32)
    return np.concatenate(score_chunks, axis=0)

def predict_brain_slice(slice_img, model_brain):
    resized_img = resize_image_slice(slice_img, target_size=IMG_TARGET)
    input_tensor = resized_img[np.newaxis, :, :, np.newaxis].astype(np.float32)

    brain_raw_output = foreground_channel(model_brain.predict(input_tensor, verbose=0))[0].astype(np.float32)
    brain_prob_map = output_to_probability_np(brain_raw_output)
    brain_pred_mask = (brain_prob_map > BRAIN_THRESHOLD).astype(np.uint8)
    roi_mask, roi_strategy, has_explainable_target = build_target_roi(brain_prob_map)
    baseline_image = make_perturbation_baseline(input_tensor[0, :, :, 0])
    base_score = compute_scalar_score_numpy(brain_raw_output, roi_mask)

    return {
        "resized_img": resized_img,
        "input_tensor": input_tensor,
        "brain_raw_output": brain_raw_output,
        "brain_prob_map": brain_prob_map,
        "brain_pred_mask": brain_pred_mask,
        "roi_mask": roi_mask.astype(np.uint8),
        "roi_strategy": roi_strategy,
        "has_explainable_target": bool(has_explainable_target),
        "baseline_image": baseline_image,
        "base_score": base_score,
    }


## 5. Métodos XAI y métricas de fidelidad

Aquí se implementan los mapas de atribución y las curvas cuantitativas. Para adaptar métodos originalmente usados en clasificación a una tarea de segmentación, se emplea un objetivo escalar basado en la salida de primer plano dentro de la región predicha.

Las curvas de Deletion, Insertion, MoRF, LeRF y el AOPC se calculan mediante perturbaciones progresivas de la imagen. Por tanto, deben interpretarse como pruebas de fidelidad respecto al comportamiento del modelo bajo ese protocolo concreto.


In [ ]:
# ============================================================
# Helper functions: XAI maps and faithfulness curves
# ============================================================

def compute_gradcam(model, input_tensor, roi_mask, target_layer_name):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(target_layer_name).output, model.output],
    )
    input_tf = tf.convert_to_tensor(input_tensor, dtype=tf.float32)
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(input_tf, training=False)
        score = segmentation_score_from_model_output(foreground_channel(predictions)[0], roi_tf)

    grads = tape.gradient(score, conv_outputs)
    if grads is None:
        raise RuntimeError("Grad-CAM gradients are None. Check the target layer and score definition.")

    conv_outputs = conv_outputs[0]
    grads = grads[0]
    weights = tf.reduce_mean(grads, axis=(0, 1))
    cam = tf.reduce_sum(conv_outputs * weights, axis=-1)
    cam = tf.nn.relu(cam)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_TARGET, IMG_TARGET), method="bilinear")[..., 0]
    return normalize_map(cam.numpy())

def compute_gradcam_pp(model, input_tensor, roi_mask, target_layer_name):
    """
    Memory-safe Grad-CAM++ approximation with conservative fallbacks.

    Exact higher-order Grad-CAM++ can be extremely memory hungry in TensorFlow for
    segmentation models. This implementation keeps a Grad-CAM++-style weighting
    scheme from first-order gradients. If positive gradients are absent, it uses
    absolute gradient magnitude as an exploratory fallback; if the resulting CAM
    is still flat, it falls back to standard Grad-CAM and emits a warning.
    """
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(target_layer_name).output, model.output],
    )
    input_tf = tf.convert_to_tensor(input_tensor, dtype=tf.float32)
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(input_tf, training=False)
        score = segmentation_score_from_model_output(foreground_channel(predictions)[0], roi_tf)

    first_grads = tape.gradient(score, conv_outputs)
    if first_grads is None:
        warnings.warn("Grad-CAM++ gradients are None; falling back to Grad-CAM.")
        return compute_gradcam(model, input_tensor, roi_mask, target_layer_name)

    conv_outputs = conv_outputs[0]
    raw_grads = first_grads[0]
    positive_grads = tf.nn.relu(raw_grads)

    if float(tf.reduce_max(positive_grads).numpy()) <= 1e-8:
        warnings.warn(
            "Grad-CAM++ positive gradients are empty; using absolute gradient "
            "magnitude as a memory-safe exploratory fallback."
        )
        positive_grads = tf.abs(raw_grads)

    grads_sq = tf.square(positive_grads)
    grads_cube = grads_sq * positive_grads

    global_sum = tf.reduce_sum(conv_outputs, axis=(0, 1), keepdims=True)
    alpha_denom = 2.0 * grads_sq + global_sum * grads_cube
    alpha_denom = tf.where(tf.abs(alpha_denom) > 1e-8, alpha_denom, tf.ones_like(alpha_denom))
    alphas = grads_sq / alpha_denom
    weights = tf.reduce_sum(alphas * positive_grads, axis=(0, 1))

    cam = tf.reduce_sum(weights * conv_outputs, axis=-1)
    cam = tf.nn.relu(cam)
    cam = tf.image.resize(cam[..., tf.newaxis], (IMG_TARGET, IMG_TARGET), method="bilinear")[..., 0]
    cam_np = cam.numpy().astype(np.float32)

    if not np.isfinite(cam_np).all() or float(np.nanmax(cam_np) - np.nanmin(cam_np)) <= 1e-8:
        warnings.warn("Grad-CAM++ produced a flat/non-finite map; falling back to Grad-CAM.")
        return compute_gradcam(model, input_tensor, roi_mask, target_layer_name)

    return normalize_map(cam_np)

def compute_integrated_gradients(model, input_tensor, roi_mask, steps=IG_STEPS, baseline_image=None):
    """Memory-safe Integrated Gradients computed incrementally."""
    input_single = tf.convert_to_tensor(input_tensor[0], dtype=tf.float32)
    if baseline_image is None:
        baseline_single = tf.zeros_like(input_single)
    else:
        baseline = np.asarray(baseline_image, dtype=np.float32)[..., np.newaxis]
        baseline_single = tf.convert_to_tensor(baseline, dtype=tf.float32)
    roi_tf = tf.convert_to_tensor(roi_mask, dtype=tf.float32)

    alphas = np.linspace(0.0, 1.0, int(steps) + 1, dtype=np.float32)
    grad_steps = []

    for alpha in alphas:
        interpolated = baseline_single + alpha * (input_single - baseline_single)
        with tf.GradientTape() as tape:
            tape.watch(interpolated)
            prediction = foreground_channel(model(interpolated[None, ...], training=False))[0]
            score = segmentation_score_from_model_output(prediction, roi_tf)
        grad = tape.gradient(score, interpolated)
        grad_steps.append(grad.numpy().astype(np.float32)[..., 0])
        del interpolated
        del prediction
        del score
        del grad

    grads = np.stack(grad_steps, axis=0)
    avg_grads = 0.5 * (grads[:-1] + grads[1:])
    mean_grads = np.mean(avg_grads, axis=0)
    integrated = (input_single.numpy()[..., 0] - baseline_single.numpy()[..., 0]) * mean_grads
    attribution = np.maximum(integrated, 0.0)
    return normalize_map(attribution)

def compute_occlusion_map(model, input_tensor, roi_mask, patch=OCCLUSION_PATCH, stride=OCCLUSION_STRIDE, baseline_image=None):
    original_image = input_tensor[0, :, :, 0].astype(np.float32)
    if float(np.sum(roi_mask)) <= 0.0:
        summary = {
            "occlusion_base_score": np.nan,
            "occlusion_mean_drop": np.nan,
            "occlusion_max_drop": np.nan,
            "occlusion_sum_drop": np.nan,
        }
        return np.zeros_like(original_image, dtype=np.float32), summary

    baseline_image = make_perturbation_baseline(original_image) if baseline_image is None else baseline_image
    base_score = compute_scalar_score_numpy(foreground_channel(model.predict(input_tensor, verbose=0))[0], roi_mask)

    height, width = original_image.shape
    y_positions = build_sliding_positions(height, patch, stride)
    x_positions = build_sliding_positions(width, patch, stride)

    perturbed_images = []
    coords = []
    for y0 in y_positions:
        y1 = min(y0 + patch, height)
        for x0 in x_positions:
            x1 = min(x0 + patch, width)
            perturbed = original_image.copy()
            perturbed[y0:y1, x0:x1] = baseline_image[y0:y1, x0:x1]
            perturbed_images.append(perturbed)
            coords.append((y0, y1, x0, x1))

    perturbed_batch = np.stack(perturbed_images, axis=0)[..., np.newaxis].astype(np.float32)
    perturbed_scores = score_batch_inputs(model, perturbed_batch, roi_mask)

    occlusion_values = np.zeros_like(original_image, dtype=np.float32)
    occlusion_counts = np.zeros_like(original_image, dtype=np.float32)
    for score_value, (y0, y1, x0, x1) in zip(perturbed_scores, coords):
        drop = float(base_score - score_value)
        occlusion_values[y0:y1, x0:x1] += drop
        occlusion_counts[y0:y1, x0:x1] += 1.0

    occlusion_map = occlusion_values / np.maximum(occlusion_counts, 1.0)
    occlusion_positive = np.maximum(occlusion_map, 0.0)
    summary = {
        "occlusion_base_score": float(base_score),
        "occlusion_mean_drop": float(np.mean(occlusion_map)),
        "occlusion_max_drop": float(np.max(occlusion_map)),
        "occlusion_sum_drop": float(np.sum(occlusion_positive)),
    }
    return normalize_map(occlusion_positive), summary

def create_curve_batch(original_image, baseline_image, sorted_indices, curve_type, curve_steps=CURVE_STEPS):
    flat_original = original_image.reshape(-1)
    flat_baseline = baseline_image.reshape(-1)
    num_pixels = flat_original.size
    counts = np.round(np.linspace(0, num_pixels, curve_steps + 1)).astype(int)
    fractions = np.linspace(0.0, 1.0, curve_steps + 1, dtype=np.float32)

    batch_images = []
    for count in counts:
        if curve_type in ("deletion", "morf", "lerf"):
            working = flat_original.copy()
            working[sorted_indices[:count]] = flat_baseline[sorted_indices[:count]]
        elif curve_type == "insertion":
            working = flat_baseline.copy()
            working[sorted_indices[:count]] = flat_original[sorted_indices[:count]]
        else:
            raise ValueError(f"Unsupported curve_type: {curve_type}")
        batch_images.append(working.reshape(original_image.shape))

    batch_inputs = np.stack(batch_images, axis=0)[..., np.newaxis].astype(np.float32)
    return fractions, batch_inputs

def trapezoid_auc(y_values, x_values):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y_values, x_values)
    return np.trapz(y_values, x_values)

def empty_curve(curve_steps=CURVE_STEPS):
    fractions = np.linspace(0.0, 1.0, int(curve_steps) + 1, dtype=np.float32)
    scores = np.full_like(fractions, np.nan, dtype=np.float32)
    return {"fractions": fractions, "scores": scores, "raw_scores": scores.copy(), "auc": np.nan}

def compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, curve_type, baseline_image=None, curve_steps=CURVE_STEPS):
    original_image = input_tensor[0, :, :, 0].astype(np.float32)
    baseline_image = make_perturbation_baseline(original_image) if baseline_image is None else baseline_image
    attribution_flat = np.asarray(attribution_map, dtype=np.float32).reshape(-1)

    descending = curve_type in ("deletion", "insertion", "morf")
    sorted_indices = np.argsort(-attribution_flat) if descending else np.argsort(attribution_flat)
    fractions, batch_inputs = create_curve_batch(original_image, baseline_image, sorted_indices, curve_type, curve_steps=curve_steps)
    raw_scores = score_batch_inputs(model, batch_inputs, roi_mask)

    original_score = raw_scores[0] if curve_type in ("deletion", "morf", "lerf") else raw_scores[-1]
    baseline_score = raw_scores[0] if curve_type == "insertion" else np.nan

    if curve_type == "insertion":
        denominator = float(original_score) - float(baseline_score)
        if not np.isfinite(denominator) or abs(denominator) < 1e-8:
            normalized_scores = np.full_like(raw_scores, np.nan, dtype=np.float32)
            auc_value = np.nan
        else:
            normalized_scores = (raw_scores - float(baseline_score)) / (denominator + 1e-8)
            auc_value = float(trapezoid_auc(normalized_scores, fractions))
    else:
        if not np.isfinite(original_score) or abs(float(original_score)) < 1e-8:
            normalized_scores = np.full_like(raw_scores, np.nan, dtype=np.float32)
            auc_value = np.nan
        else:
            normalized_scores = raw_scores / (float(original_score) + 1e-8)
            auc_value = float(trapezoid_auc(normalized_scores, fractions))

    return {
        "fractions": fractions.astype(np.float32),
        "scores": normalized_scores.astype(np.float32),
        "raw_scores": raw_scores.astype(np.float32),
        "original_score": float(original_score) if np.isfinite(original_score) else np.nan,
        "baseline_score": float(baseline_score) if np.isfinite(baseline_score) else np.nan,
        "auc": auc_value,
    }

def assert_faithfulness_curve_sanity(faithfulness_bundle, context="faithfulness", endpoint_tol=5e-3, auc_tol=1e-6):
    """Validate normalized curves used for quantitative XAI metrics."""
    deletion = np.asarray(faithfulness_bundle["deletion"]["scores"], dtype=np.float32)
    insertion = np.asarray(faithfulness_bundle["insertion"]["scores"], dtype=np.float32)

    if np.isfinite(deletion).any() and not np.isclose(deletion[0], 1.0, atol=endpoint_tol):
        raise AssertionError(f"{context}: Deletion curve should start near 1, got {deletion[0]:.6f}")
    if np.isfinite(insertion).any():
        if not np.isclose(insertion[0], 0.0, atol=endpoint_tol):
            raise AssertionError(f"{context}: Insertion curve should start near 0, got {insertion[0]:.6f}")
        if not np.isclose(insertion[-1], 1.0, atol=endpoint_tol):
            raise AssertionError(f"{context}: Insertion curve should end near 1, got {insertion[-1]:.6f}")

    for curve_name in ("deletion", "insertion", "morf", "lerf"):
        auc_value = faithfulness_bundle[curve_name]["auc"]
        if np.isfinite(auc_value) and auc_value < -auc_tol:
            raise AssertionError(f"{context}: {curve_name} AUC should be non-negative, got {auc_value:.6f}")

def compute_faithfulness_bundle(model, input_tensor, roi_mask, attribution_map, baseline_image=None, curve_steps=CURVE_STEPS):
    if float(np.sum(roi_mask)) <= 0.0:
        return {
            "deletion": empty_curve(curve_steps),
            "insertion": empty_curve(curve_steps),
            "morf": empty_curve(curve_steps),
            "lerf": empty_curve(curve_steps),
            "aopc": np.nan,
            "normalized_aopc": np.nan,
        }

    baseline_image = make_perturbation_baseline(input_tensor[0, :, :, 0]) if baseline_image is None else baseline_image
    deletion = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "deletion", baseline_image=baseline_image, curve_steps=curve_steps)
    insertion = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "insertion", baseline_image=baseline_image, curve_steps=curve_steps)
    morf = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "morf", baseline_image=baseline_image, curve_steps=curve_steps)
    lerf = compute_curve_from_attribution(model, input_tensor, roi_mask, attribution_map, "lerf", baseline_image=baseline_image, curve_steps=curve_steps)

    base_score = float(morf["raw_scores"][0])
    raw_drops = base_score - morf["raw_scores"][1:]
    aopc = float(np.nanmean(raw_drops)) if len(raw_drops) > 0 else np.nan
    normalized_aopc = float(np.nanmean(1.0 - morf["scores"][1:])) if len(morf["scores"]) > 1 else np.nan

    faithfulness_bundle = {
        "deletion": deletion,
        "insertion": insertion,
        "morf": morf,
        "lerf": lerf,
        "aopc": aopc,
        "normalized_aopc": normalized_aopc,
    }
    assert_faithfulness_curve_sanity(faithfulness_bundle)
    return faithfulness_bundle

def compute_xai_bundle(
    model,
    input_tensor,
    roi_mask,
    target_layer_name,
    baseline_image=None,
    ig_steps=IG_STEPS,
    curve_steps=CURVE_STEPS,
    occlusion_patch=OCCLUSION_PATCH,
    occlusion_stride=OCCLUSION_STRIDE,
    include_occlusion=True,
    faithfulness_methods=None,
):
    baseline_image = make_perturbation_baseline(input_tensor[0, :, :, 0]) if baseline_image is None else baseline_image

    if float(np.sum(roi_mask)) <= 0.0:
        attribution_maps = {
            "gradcam": np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32),
            "gradcampp": np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32),
            "ig": np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32),
        }
    else:
        attribution_maps = {
            "gradcam": compute_gradcam(model, input_tensor, roi_mask, target_layer_name),
            "gradcampp": compute_gradcam_pp(model, input_tensor, roi_mask, target_layer_name),
            "ig": compute_integrated_gradients(model, input_tensor, roi_mask, steps=ig_steps, baseline_image=baseline_image),
        }

    if include_occlusion:
        occlusion_map, occlusion_summary = compute_occlusion_map(
            model,
            input_tensor,
            roi_mask,
            patch=occlusion_patch,
            stride=occlusion_stride,
            baseline_image=baseline_image,
        )
    else:
        occlusion_map = np.zeros((IMG_TARGET, IMG_TARGET), dtype=np.float32)
        occlusion_summary = {
            "occlusion_base_score": np.nan,
            "occlusion_mean_drop": np.nan,
            "occlusion_max_drop": np.nan,
            "occlusion_sum_drop": np.nan,
        }

    faithfulness_methods = tuple(attribution_maps.keys()) if faithfulness_methods is None else tuple(faithfulness_methods)
    faithfulness_by_method = {}
    for method_name, attribution_map in attribution_maps.items():
        if method_name not in faithfulness_methods:
            continue
        faithfulness_by_method[method_name] = compute_faithfulness_bundle(
            model,
            input_tensor,
            roi_mask,
            attribution_map,
            baseline_image=baseline_image,
            curve_steps=curve_steps,
        )
        assert_faithfulness_curve_sanity(faithfulness_by_method[method_name], context=f"{method_name}")

    return {
        "attribution_maps": attribution_maps,
        "occlusion_map": occlusion_map,
        "occlusion_summary": occlusion_summary,
        "faithfulness_by_method": faithfulness_by_method,
    }

def assert_normalized_map(name, array_2d):
    arr = np.asarray(array_2d, dtype=np.float32)
    if arr.shape != (IMG_TARGET, IMG_TARGET):
        raise AssertionError(f"{name} must have shape {(IMG_TARGET, IMG_TARGET)}, got {arr.shape}")
    if not np.isfinite(arr).all():
        raise AssertionError(f"{name} contains non-finite values")
    if float(arr.min()) < -1e-6 or float(arr.max()) > 1.0 + 1e-6:
        raise AssertionError(f"{name} is not normalized to [0, 1]")


## 6. Selección de ejemplos, visualización y tablas

Estas funciones organizan la selección de cortes representativos y la exportación de resultados. Se incluyen grupos de calidad de segmentación, cortes interiores y cortes extremos para inspeccionar posibles efectos de borde.

El patrón visual generado está pensado para la memoria: imagen original, referencia manual, predicción del modelo y superposiciones de XAI, acompañadas de curvas de fidelidad cuando procede.


In [ ]:
# ============================================================
# Helper functions: selection, plotting, and export tables
# ============================================================

def collect_smoke_examples(dataset_dir):
    interior_example = None
    edge_example = None
    small_area_example = None

    for case_dir in sort_case_dirs(dataset_dir):
        nifti_path = case_dir / "Seq No.nii"
        if not nifti_path.exists():
            continue
        _, volume = load_nifti_volume(nifti_path)
        brain_masks = load_union_mask_dict(case_dir / "cerebro", volume.shape)
        total_slices = int(volume.shape[2])

        candidates = []
        for slice_idx in range(total_slices):
            slice_img = volume[:, :, slice_idx]
            gt_mask = resize_mask(brain_masks.get(slice_idx, np.zeros_like(slice_img, dtype=np.uint8)))
            if gt_mask.sum() == 0:
                continue
            example = {
                "case": case_dir.name,
                "slice_idx": slice_idx,
                "slice_img": slice_img,
                "gt_mask": gt_mask,
                "total_slices_case": total_slices,
            }
            candidates.append(example)
            if get_edge_position_label(slice_idx, total_slices) is None and interior_example is None:
                interior_example = example
            if get_edge_position_label(slice_idx, total_slices) is not None and edge_example is None:
                edge_example = example

        if candidates and small_area_example is None:
            small_area_example = min(candidates, key=lambda item: int(item["gt_mask"].sum()))
        if interior_example is not None and edge_example is not None and small_area_example is not None:
            return interior_example, edge_example, small_area_example

    return interior_example, edge_example, small_area_example

def pick_quantile_rows(df, column, quantiles):
    if df.empty:
        return []
    ordered = df.sort_values(column).reset_index(drop=False)
    chosen = []
    for quantile in quantiles:
        target_value = ordered[column].quantile(quantile)
        idx = (ordered[column] - target_value).abs().idxmin()
        original_idx = int(ordered.loc[idx, "index"])
        if original_idx not in chosen:
            chosen.append(original_idx)
    return chosen

def select_quality_slices(slice_summary_df, group_name, lower_q, upper_q, limit=SELECTED_SLICES_PER_GROUP):
    subset = slice_summary_df[slice_summary_df["has_explainable_target"]].copy()
    if subset.empty:
        return []
    lower = subset["brain_dice"].quantile(lower_q)
    upper = subset["brain_dice"].quantile(upper_q)
    quality_subset = subset[(subset["brain_dice"] >= lower) & (subset["brain_dice"] <= upper)].copy()
    if quality_subset.empty:
        quality_subset = subset.copy()
    selected_indices = pick_quantile_rows(quality_subset, "brain_dice", [0.20, 0.50, 0.80])
    backfill = quality_subset.sort_values(["brain_dice", "gt_brain_pixels"], ascending=[upper_q < 0.5, False]).index.tolist()
    for idx in backfill:
        if idx not in selected_indices:
            selected_indices.append(int(idx))
        if len(selected_indices) >= limit:
            break
    return [
        {
            "slice_uid": slice_summary_df.loc[idx, "slice_uid"],
            "case": slice_summary_df.loc[idx, "case"],
            "slice_idx_original": int(slice_summary_df.loc[idx, "slice_idx_original"]),
            "selection_group": group_name,
            "selection_reason": "brain_dice_quantile",
        }
        for idx in selected_indices[:limit]
    ]

def select_area_slices(slice_summary_df, column, group_name, limit=SELECTED_SLICES_PER_GROUP):
    subset = slice_summary_df[slice_summary_df["has_explainable_target"]].copy()
    if subset.empty:
        return []
    selected = subset.sort_values([column, "brain_dice"], ascending=[True, True]).head(limit)
    return [
        {
            "slice_uid": row["slice_uid"],
            "case": row["case"],
            "slice_idx_original": int(row["slice_idx_original"]),
            "selection_group": group_name,
            "selection_reason": f"lowest_{column}",
        }
        for _, row in selected.iterrows()
    ]

def select_position_slices(slice_summary_df, group_name, is_edge, limit=SELECTED_SLICES_PER_GROUP):
    subset = slice_summary_df[slice_summary_df["is_edge_first_last3"].fillna(False) == bool(is_edge)].copy()
    if subset.empty:
        return []
    sort_cols = ["has_explainable_target", "gt_brain_pixels", "brain_dice"]
    selected = subset.sort_values(sort_cols, ascending=[False, False, False]).head(limit)
    return [
        {
            "slice_uid": row["slice_uid"],
            "case": row["case"],
            "slice_idx_original": int(row["slice_idx_original"]),
            "selection_group": group_name,
            "selection_reason": "position_representative",
        }
        for _, row in selected.iterrows()
    ]

def build_selected_slices(slice_summary_df):
    selected_records = []
    selected_records.extend(select_quality_slices(slice_summary_df, "low_brain_segmentation", 0.00, 0.33))
    selected_records.extend(select_quality_slices(slice_summary_df, "medium_brain_segmentation", 0.33, 0.66))
    selected_records.extend(select_quality_slices(slice_summary_df, "good_brain_segmentation", 0.66, 1.00))
    selected_records.extend(select_area_slices(slice_summary_df, "brain_area_fraction_gt", "low_gt_brain_area"))
    selected_records.extend(select_area_slices(slice_summary_df, "brain_area_fraction_pred", "low_pred_brain_area"))
    selected_records.extend(select_position_slices(slice_summary_df, "interior", is_edge=False))
    selected_records.extend(select_position_slices(slice_summary_df, "edge_first_last3", is_edge=True))
    if not selected_records:
        return pd.DataFrame(columns=["slice_uid", "case", "slice_idx_original", "selection_group", "selection_reason"])
    return pd.DataFrame(selected_records).drop_duplicates(["slice_uid", "selection_group"])

def make_export_filename(case_name, slice_idx_original):
    safe_case = re.sub(r"[^A-Za-z0-9_\-]", "_", case_name)
    return f"{safe_case}_slice_{int(slice_idx_original):03d}.png"

def plot_heatmap_overlay(ax, image_2d, heatmap_2d, title, cmap="jet"):
    ax.imshow(image_2d, cmap="gray")
    ax.imshow(heatmap_2d, cmap=cmap, alpha=0.45, vmin=0.0, vmax=1.0)
    ax.set_title(title)
    ax.axis("off")

def render_slice_montage(artifact, output_path):
    image = artifact["image"]
    gt_mask = artifact["gt_mask"]
    pred_mask = artifact["pred_mask"]
    gradcam = artifact["gradcam"]
    gradcampp = artifact["gradcampp"]
    ig_map = artifact["ig"]
    faithfulness = artifact["faithfulness_by_method"]
    meta = artifact["meta"]

    fig, axes = plt.subplots(1, 7, figsize=(28, 4.5))

    axes[0].imshow(image, cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(gt_mask, cmap="gray")
    axes[1].set_title("GT cerebro")
    axes[1].axis("off")

    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Pred cerebro")
    axes[2].axis("off")

    plot_heatmap_overlay(axes[3], image, gradcam, "Grad-CAM")
    plot_heatmap_overlay(axes[4], image, gradcampp, "Grad-CAM++")
    plot_heatmap_overlay(axes[5], image, ig_map, "Integrated Gradients")

    if meta.get("has_explainable_target", meta.get("pred_brain_pixels", 0) > 0):
        for method_name, label in (("gradcam", "Grad-CAM"), ("gradcampp", "Grad-CAM++"), ("ig", "IG")):
            curve = faithfulness[method_name]["deletion"]
            axes[6].plot(
                curve["fractions"],
                curve["scores"],
                label=f"{label} (AUC={curve['auc']:.3f})",
                linewidth=2,
            )
        axes[6].set_title("Deletion normalizada")
        axes[6].set_xlabel("Perturbed fraction")
        axes[6].set_ylabel("Score / original score")
        axes[6].set_ylim(bottom=0.0)
        axes[6].grid(alpha=0.25)
        axes[6].legend(fontsize=8)
    else:
        axes[6].axis("off")
        axes[6].text(0.5, 0.5, "No predicted brain\nNo quantitative XAI target", ha="center", va="center")

    figure_title = (
        f"{meta['case']} | slice {meta['slice_idx_original']} | "
        f"Dice={meta['brain_dice']:.3f} | GT px={meta['gt_brain_pixels']} | "
        f"Pred px={meta['pred_brain_pixels']} | area ratio={meta['brain_area_ratio_pred_gt']:.3f} | "
        f"target={meta['target_mask_strategy_used']}"
    )
    fig.suptitle(figure_title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

def get_curve_record_columns():
    return [
        "slice_uid",
        "case",
        "slice_idx_original",
        "method",
        "curve_type",
        "step_idx",
        "fraction",
        "score",
        "raw_score",
    ]

def build_group_summary(method_metrics_df, slice_summary_df):
    numeric_columns = [
        "deletion_auc",
        "insertion_auc",
        "morf_auc",
        "lerf_auc",
        "aopc",
        "normalized_aopc",
        "occlusion_mean_drop",
        "occlusion_max_drop",
        "occlusion_sum_drop",
    ]
    merge_columns = [
        "slice_uid",
        "is_edge_first_last3",
        "is_interior",
        "has_explainable_target",
        "brain_quality_group",
        "low_gt_brain_area",
        "low_pred_brain_area",
    ]
    merged = method_metrics_df.merge(slice_summary_df[merge_columns], on="slice_uid", how="left")
    group_filters = {
        "edge_first_last3": merged["is_edge_first_last3"].fillna(False),
        "interior": merged["is_interior"].fillna(False),
        "good_brain_segmentation": merged["brain_quality_group"].eq("good"),
        "medium_brain_segmentation": merged["brain_quality_group"].eq("medium"),
        "low_brain_segmentation": merged["brain_quality_group"].eq("low"),
        "low_gt_brain_area": merged["low_gt_brain_area"].fillna(False),
        "low_pred_brain_area": merged["low_pred_brain_area"].fillna(False),
        "quantitative_xai_included": merged["has_explainable_target"].fillna(False),
    }
    rows = []
    for group_name, group_mask in group_filters.items():
        group_subset = merged[group_mask]
        if group_subset.empty:
            continue
        for method_name, method_subset in group_subset.groupby("method"):
            for metric_name in numeric_columns:
                metric_values = method_subset[metric_name].dropna()
                if metric_values.empty:
                    continue
                rows.append({
                    "group": group_name,
                    "method": method_name,
                    "metric": metric_name,
                    "n": int(metric_values.shape[0]),
                    "mean": float(metric_values.mean()),
                    "std": float(metric_values.std(ddof=0)),
                    "median": float(metric_values.median()),
                })
    return pd.DataFrame(rows)

def save_checkpoint_tables(slice_records, method_records, curve_records, curve_records_path=None):
    pd.DataFrame(slice_records).to_csv(CSV_DIR / "slice_summary_checkpoint.csv", index=False)
    pd.DataFrame(method_records).to_csv(CSV_DIR / "method_metrics_checkpoint.csv", index=False)
    curve_path = Path(curve_records_path) if curve_records_path is not None else CSV_DIR / "curve_points_checkpoint.csv"
    pd.DataFrame(curve_records, columns=get_curve_record_columns()).to_csv(curve_path, index=False)

def load_checkpoint_tables(slice_path=None, method_path=None, curve_path=None):
    slice_path = Path(slice_path) if slice_path is not None else CSV_DIR / "slice_summary_checkpoint.csv"
    method_path = Path(method_path) if method_path is not None else CSV_DIR / "method_metrics_checkpoint.csv"
    curve_path = Path(curve_path) if curve_path is not None else CSV_DIR / "curve_points_checkpoint.csv"

    if not slice_path.exists():
        raise FileNotFoundError(f"Missing slice checkpoint table: {slice_path}")
    if not method_path.exists():
        raise FileNotFoundError(f"Missing method checkpoint table: {method_path}")

    slice_summary_df = pd.read_csv(slice_path)
    method_metrics_df = pd.read_csv(method_path)
    if curve_path.exists():
        curve_points_df = pd.read_csv(curve_path)
    else:
        curve_points_df = pd.DataFrame(columns=get_curve_record_columns())
    if "raw_score" not in curve_points_df.columns:
        curve_points_df["raw_score"] = curve_points_df["score"] if "score" in curve_points_df.columns else np.nan
    return slice_summary_df, method_metrics_df, curve_points_df


## 7. Carga del modelo y resolución de la capa objetivo

Se carga el modelo de segmentación cerebral y se determina la capa convolucional usada para Grad-CAM y Grad-CAM++. Si no se especifica manualmente, el cuaderno selecciona automáticamente una capa compatible, preferentemente cercana a la salida de segmentación.


In [ ]:
model_brain, resolved_target_layer, detected_brain_outputs_logits = load_brain_model(
    MODEL_BRAIN_PATH,
    target_layer_name=TARGET_LAYER_NAME,
)
BRAIN_MODEL_OUTPUTS_LOGITS = bool(detected_brain_outputs_logits)

case_dirs = sort_case_dirs(DATASET_DIR)
if not case_dirs:
    raise FileNotFoundError(f"No CASE_* directories were found under {DATASET_DIR}")

print(f"Loaded {len(case_dirs)} cases")
print(f"Resolved brain target layer: {resolved_target_layer}")
print(f"Brain model outputs logits: {BRAIN_MODEL_OUTPUTS_LOGITS}")


## 8. Celda opcional de depuración de Grad-CAM++

Esta celda está pensada para diagnosticar de forma rápida un único corte si Grad-CAM++ produce mapas vacíos o poco informativos. No forma parte obligatoria del análisis completo, pero es útil para dejar trazabilidad cuando se ajusta la capa objetivo o se revisan problemas de memoria.


In [ ]:
# ============================================================
# Optional fast Grad-CAM++ debug cell
# ============================================================
# Run this cell after loading the helpers and model if Grad-CAM++ looks empty.
# It analyzes one slice only and skips occlusion/Deletion/Insertion, so it should
# be much faster than the full notebook.

DEBUG_CASE = "CASE_20"  # Example: "CASE_1". None = first available CASE_*.
DEBUG_SLICE_IDX = "12"  # Example: 12. None = first slice with a GT brain mask.
DEBUG_TARGET_LAYER_NAME = resolved_target_layer

def find_debug_brain_example(dataset_dir, case_name=None, slice_idx=None):
    candidate_cases = sort_case_dirs(dataset_dir)
    if case_name is not None:
        candidate_cases = [Path(dataset_dir) / case_name]

    for case_dir in candidate_cases:
        nifti_path = case_dir / "Seq No.nii"
        if not nifti_path.exists():
            continue
        _, volume = load_nifti_volume(nifti_path)
        brain_masks = load_union_mask_dict(case_dir / "cerebro", volume.shape)
        slice_indices = [int(slice_idx)] if slice_idx is not None else range(volume.shape[2])
        for current_idx in slice_indices:
            if current_idx < 0 or current_idx >= volume.shape[2]:
                continue
            slice_img = volume[:, :, current_idx]
            gt_mask = resize_mask(brain_masks.get(current_idx, np.zeros_like(slice_img, dtype=np.uint8)))
            if slice_idx is None and gt_mask.sum() == 0:
                continue
            return {
                "case": case_dir.name,
                "slice_idx": int(current_idx),
                "slice_img": slice_img,
                "gt_mask": gt_mask,
            }
    raise RuntimeError("No debug slice found. Set DEBUG_CASE and DEBUG_SLICE_IDX manually.")

def debug_gradcampp_slice(model, example, target_layer_name):
    prediction = predict_brain_slice(example["slice_img"], model_brain=model)
    input_tf = tf.convert_to_tensor(prediction["input_tensor"], dtype=tf.float32)
    roi_tf = tf.convert_to_tensor(prediction["roi_mask"], dtype=tf.float32)
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(target_layer_name).output, model.output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, model_outputs = grad_model(input_tf, training=False)
        foreground = foreground_channel(model_outputs)[0]
        score = segmentation_score_from_model_output(foreground, roi_tf)
    grads = tape.gradient(score, conv_outputs)

    debug = {
        "case": example["case"],
        "slice_idx": int(example["slice_idx"]),
        "target_layer": target_layer_name,
        "gt_brain_pixels": int(example["gt_mask"].sum()),
        "pred_brain_pixels": int(prediction["brain_pred_mask"].sum()),
        "roi_pixels": int(prediction["roi_mask"].sum()),
        "base_score": float(prediction["base_score"]),
        "tape_score": float(score.numpy()),
        "conv_shape": tuple(conv_outputs.shape),
        "output_shape": tuple(first_model_output(model_outputs).shape),
        "grads_is_none": grads is None,
    }

    if grads is not None:
        grads_np = grads.numpy().astype(np.float32)
        positive_np = np.maximum(grads_np, 0.0)
        abs_np = np.abs(grads_np)
        debug.update({
            "grad_min": float(np.min(grads_np)),
            "grad_max": float(np.max(grads_np)),
            "grad_mean": float(np.mean(grads_np)),
            "grad_abs_max": float(np.max(abs_np)),
            "positive_grad_pixels": int(np.sum(positive_np > 0.0)),
            "nonzero_grad_pixels": int(np.sum(abs_np > 0.0)),
        })

    gradcam_map = compute_gradcam(model, prediction["input_tensor"], prediction["roi_mask"], target_layer_name)
    gradcampp_map = compute_gradcam_pp(model, prediction["input_tensor"], prediction["roi_mask"], target_layer_name)
    debug.update({
        "gradcam_min": float(np.min(gradcam_map)),
        "gradcam_max": float(np.max(gradcam_map)),
        "gradcam_sum": float(np.sum(gradcam_map)),
        "gradcampp_min": float(np.min(gradcampp_map)),
        "gradcampp_max": float(np.max(gradcampp_map)),
        "gradcampp_sum": float(np.sum(gradcampp_map)),
    })

    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(prediction["resized_img"], cmap="gray")
    axes[0].set_title("Original")
    axes[1].imshow(example["gt_mask"], cmap="gray")
    axes[1].set_title("GT cerebro")
    axes[2].imshow(prediction["brain_pred_mask"], cmap="gray")
    axes[2].set_title("Pred cerebro")
    plot_heatmap_overlay(axes[3], prediction["resized_img"], gradcam_map, "Grad-CAM")
    plot_heatmap_overlay(axes[4], prediction["resized_img"], gradcampp_map, "Grad-CAM++ debug")
    for ax in axes[:3]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

    return debug

debug_example = find_debug_brain_example(DATASET_DIR, DEBUG_CASE, DEBUG_SLICE_IDX)
debug_info = debug_gradcampp_slice(model_brain, debug_example, DEBUG_TARGET_LAYER_NAME)
pd.DataFrame([debug_info]).T.rename(columns={0: "value"})


## 9. Pruebas rápidas de funcionamiento

Antes de procesar todo el conjunto de datos, se ejecutan pruebas reducidas sobre ejemplos representativos. Esta fase comprueba que los mapas estén normalizados, que las curvas tengan el número esperado de puntos y que no aparezcan valores no finitos en cortes explicables.


In [ ]:
# ============================================================
# Smoke tests
# ============================================================

interior_example, edge_example, small_area_example = collect_smoke_examples(DATASET_DIR)
smoke_examples = [
    ("interior_brain", interior_example),
    ("edge_first_last3", edge_example),
    ("small_gt_brain_area", small_area_example),
]
smoke_rows = []

for example_name, example in smoke_examples:
    if example is None:
        print(f"Smoke test skipped: no {example_name} example found in {DATASET_DIR}")
        continue

    prediction = predict_brain_slice(example["slice_img"], model_brain=model_brain)
    xai_bundle = compute_xai_bundle(
        model=model_brain,
        input_tensor=prediction["input_tensor"],
        roi_mask=prediction["roi_mask"],
        target_layer_name=resolved_target_layer,
        baseline_image=prediction["baseline_image"],
        ig_steps=SMOKE_TEST_IG_STEPS,
        curve_steps=SMOKE_TEST_CURVE_STEPS,
        occlusion_patch=SMOKE_TEST_OCCLUSION_PATCH,
        occlusion_stride=SMOKE_TEST_OCCLUSION_STRIDE,
        include_occlusion=True,
        faithfulness_methods=SMOKE_TEST_CURVE_METHODS,
    )

    for map_name, attribution_map in xai_bundle["attribution_maps"].items():
        assert_normalized_map(f"{example_name}:{map_name}", attribution_map)
    assert_normalized_map(f"{example_name}:occlusion", xai_bundle["occlusion_map"])

    for method_name in SMOKE_TEST_CURVE_METHODS:
        faithfulness = xai_bundle["faithfulness_by_method"][method_name]
        for curve_name in ("deletion", "insertion", "morf", "lerf"):
            scores = faithfulness[curve_name]["scores"]
            fractions = faithfulness[curve_name]["fractions"]
            if len(scores) != SMOKE_TEST_CURVE_STEPS + 1 or len(fractions) != SMOKE_TEST_CURVE_STEPS + 1:
                raise AssertionError(f"Unexpected number of curve points for {example_name}:{method_name}:{curve_name}")
            if prediction["has_explainable_target"] and not np.isfinite(scores).all():
                raise AssertionError(f"Non-finite scores in {example_name}:{method_name}:{curve_name}")
        if prediction["has_explainable_target"]:
            assert_faithfulness_curve_sanity(faithfulness, context=f"smoke:{example_name}:{method_name}")

    gradcam_faithfulness = xai_bundle["faithfulness_by_method"][SMOKE_TEST_CURVE_METHODS[0]]
    smoke_rows.append({
        "example_type": example_name,
        "case": example["case"],
        "slice_idx": example["slice_idx"],
        "gt_brain_pixels": int(example["gt_mask"].sum()),
        "pred_brain_pixels": int(prediction["brain_pred_mask"].sum()),
        "roi_strategy": prediction["roi_strategy"],
        "has_explainable_target": bool(prediction["has_explainable_target"]),
        "target_layer": resolved_target_layer,
        "gradcam_min": float(xai_bundle["attribution_maps"]["gradcam"].min()),
        "gradcam_max": float(xai_bundle["attribution_maps"]["gradcam"].max()),
        "deletion_points": len(gradcam_faithfulness["deletion"]["scores"]),
        "insertion_delta": float(gradcam_faithfulness["insertion"]["scores"][-1] - gradcam_faithfulness["insertion"]["scores"][0]),
        "deletion_delta": float(gradcam_faithfulness["deletion"]["scores"][0] - gradcam_faithfulness["deletion"]["scores"][-1]),
    })

    del prediction
    del xai_bundle
    gc.collect()

smoke_df = pd.DataFrame(smoke_rows)
print("Smoke tests finished")
print("This smoke test uses reduced-memory settings on purpose.")
smoke_df


## 10. Análisis completo del conjunto de datos

En esta fase se recorren todos los casos y cortes disponibles. Para cada corte se calcula la predicción cerebral, las métricas Dice e IoU frente a la máscara manual, los mapas XAI y las métricas de fidelidad.

Los resultados se van guardando progresivamente para reducir el riesgo de pérdida de información en ejecuciones largas.


In [ ]:
# ============================================================
# Full dataset analysis
# ============================================================

slice_records = []
method_metric_records = []
curve_records = []

def assign_brain_quality_group(dice_value):
    if dice_value >= 0.90:
        return "good"
    if dice_value >= 0.75:
        return "medium"
    return "low"

for case_dir in case_dirs:
    nifti_path = case_dir / "Seq No.nii"
    if not nifti_path.exists():
        print(f"Skipping {case_dir.name}: missing Seq No.nii")
        continue

    nii, volume = load_nifti_volume(nifti_path)
    total_slices = int(volume.shape[2])
    brain_masks = load_union_mask_dict(case_dir / "cerebro", volume.shape)

    print(f"Processing {case_dir.name}: {total_slices} slices")

    for slice_idx in range(total_slices):
        slice_img = volume[:, :, slice_idx]
        gt_mask_raw = brain_masks.get(slice_idx, np.zeros_like(slice_img, dtype=np.uint8))
        gt_mask = resize_mask(gt_mask_raw, target_size=IMG_TARGET)

        prediction = predict_brain_slice(slice_img, model_brain=model_brain)
        xai_bundle = compute_xai_bundle(
            model=model_brain,
            input_tensor=prediction["input_tensor"],
            roi_mask=prediction["roi_mask"],
            target_layer_name=resolved_target_layer,
            baseline_image=prediction["baseline_image"],
            ig_steps=IG_STEPS,
            curve_steps=CURVE_STEPS,
            occlusion_patch=OCCLUSION_PATCH,
            occlusion_stride=OCCLUSION_STRIDE,
            include_occlusion=True,
            faithfulness_methods=None,
        )

        slice_uid = f"{case_dir.name}__slice_{slice_idx:04d}"
        edge_label = get_edge_position_label(slice_idx, total_slices)
        total_pixels = int(IMG_TARGET * IMG_TARGET)
        gt_brain_pixels = int(gt_mask.sum())
        pred_brain_pixels = int(prediction["brain_pred_mask"].sum())
        brain_dice = float(dice_numpy(gt_mask, prediction["brain_pred_mask"]))
        brain_iou = float(iou_numpy(gt_mask, prediction["brain_pred_mask"]))
        brain_area_ratio_pred_gt = float(pred_brain_pixels / (gt_brain_pixels + 1e-8))
        brain_area_fraction_pred = float(pred_brain_pixels / total_pixels)
        brain_area_fraction_gt = float(gt_brain_pixels / total_pixels)

        slice_record = {
            "slice_uid": slice_uid,
            "case": case_dir.name,
            "slice_idx_original": int(slice_idx),
            "total_slices_case": total_slices,
            "edge_position_label": edge_label,
            "is_edge_first_last3": bool(edge_label is not None),
            "is_interior": bool(edge_label is None),
            "gt_brain_pixels": gt_brain_pixels,
            "pred_brain_pixels": pred_brain_pixels,
            "brain_area_ratio_pred_gt": brain_area_ratio_pred_gt,
            "brain_area_fraction_pred": brain_area_fraction_pred,
            "brain_area_fraction_gt": brain_area_fraction_gt,
            "target_mask_strategy_used": prediction["roi_strategy"],
            "has_explainable_target": bool(prediction["has_explainable_target"]),
            "base_score": float(prediction["base_score"]),
            "brain_dice": brain_dice,
            "brain_iou": brain_iou,
            "brain_quality_group": assign_brain_quality_group(brain_dice),
            "target_layer_name_used": resolved_target_layer,
            "selected_for_export": False,
            "selected_export_groups": "",
            "selection_reasons": "",
            "figure_paths": "",
        }
        slice_records.append(slice_record)

        for method_name, faithfulness in xai_bundle["faithfulness_by_method"].items():
            method_metric_records.append({
                "slice_uid": slice_uid,
                "case": case_dir.name,
                "slice_idx_original": int(slice_idx),
                "method": method_name,
                "deletion_auc": float(faithfulness["deletion"]["auc"]),
                "insertion_auc": float(faithfulness["insertion"]["auc"]),
                "morf_auc": float(faithfulness["morf"]["auc"]),
                "lerf_auc": float(faithfulness["lerf"]["auc"]),
                "aopc": float(faithfulness["aopc"]),
                "normalized_aopc": float(faithfulness["normalized_aopc"]),
                "occlusion_mean_drop": np.nan,
                "occlusion_max_drop": np.nan,
                "occlusion_sum_drop": np.nan,
                "figure_paths": "",
            })

            for curve_name in ("deletion", "insertion", "morf", "lerf"):
                curve_bundle = faithfulness[curve_name]
                for step_idx, (fraction, score_value, raw_score_value) in enumerate(zip(curve_bundle["fractions"], curve_bundle["scores"], curve_bundle["raw_scores"])):
                    curve_records.append({
                        "slice_uid": slice_uid,
                        "case": case_dir.name,
                        "slice_idx_original": int(slice_idx),
                        "method": method_name,
                        "curve_type": curve_name,
                        "step_idx": int(step_idx),
                        "fraction": float(fraction),
                        "score": float(score_value),
                        "raw_score": float(raw_score_value),
                    })

        occlusion_summary = xai_bundle["occlusion_summary"]
        method_metric_records.append({
            "slice_uid": slice_uid,
            "case": case_dir.name,
            "slice_idx_original": int(slice_idx),
            "method": "occlusion",
            "deletion_auc": np.nan,
            "insertion_auc": np.nan,
            "morf_auc": np.nan,
            "lerf_auc": np.nan,
            "aopc": np.nan,
            "normalized_aopc": np.nan,
            "occlusion_mean_drop": float(occlusion_summary["occlusion_mean_drop"]),
            "occlusion_max_drop": float(occlusion_summary["occlusion_max_drop"]),
            "occlusion_sum_drop": float(occlusion_summary["occlusion_sum_drop"]),
            "figure_paths": "",
        })

        del prediction
        del xai_bundle
        gc.collect()

    save_checkpoint_tables(slice_records, method_metric_records, curve_records)
    del nii
    del volume
    del brain_masks
    gc.collect()

slice_summary_preview = pd.DataFrame(slice_records)
if not slice_summary_preview.empty:
    gt_cutoff = slice_summary_preview["brain_area_fraction_gt"].quantile(LOW_BRAIN_AREA_FRACTION_QUANTILE)
    pred_cutoff = slice_summary_preview["brain_area_fraction_pred"].quantile(SMALL_PRED_BRAIN_AREA_FRACTION_QUANTILE)
    slice_summary_preview["low_gt_brain_area"] = slice_summary_preview["brain_area_fraction_gt"] <= gt_cutoff
    slice_summary_preview["low_pred_brain_area"] = slice_summary_preview["brain_area_fraction_pred"] <= pred_cutoff
    for record, (_, enriched_row) in zip(slice_records, slice_summary_preview.iterrows()):
        record["low_gt_brain_area"] = bool(enriched_row["low_gt_brain_area"])
        record["low_pred_brain_area"] = bool(enriched_row["low_pred_brain_area"])
    save_checkpoint_tables(slice_records, method_metric_records, curve_records)

print("Heavy analysis finished. Checkpoint tables saved in:")
print(f"  - {CSV_DIR / 'slice_summary_checkpoint.csv'}")
print(f"  - {CSV_DIR / 'method_metrics_checkpoint.csv'}")
print(f"  - {CSV_DIR / 'curve_points_checkpoint.csv'}")
print("Run the next cell to reload checkpoint tables without repeating inference/XAI.")


## 11. Selección de cortes y exportación de figuras

Tras generar las tablas de métricas, el cuaderno selecciona cortes representativos para revisión cualitativa. Se priorizan ejemplos con diferente calidad de segmentación, regiones cerebrales pequeñas, cortes interiores y cortes en los extremos del volumen.

Los mapas se recalculan solo para los cortes seleccionados, lo que reduce el consumo de memoria sin modificar los resultados tabulares ya obtenidos.


In [ ]:
# ============================================================
# Reload checkpoint tables, then slice selection and figure export
# ============================================================

slice_summary_df, method_metrics_df, curve_points_df = load_checkpoint_tables()
if "low_gt_brain_area" not in slice_summary_df.columns:
    gt_cutoff = slice_summary_df["brain_area_fraction_gt"].quantile(LOW_BRAIN_AREA_FRACTION_QUANTILE)
    slice_summary_df["low_gt_brain_area"] = slice_summary_df["brain_area_fraction_gt"] <= gt_cutoff
if "low_pred_brain_area" not in slice_summary_df.columns:
    pred_cutoff = slice_summary_df["brain_area_fraction_pred"].quantile(SMALL_PRED_BRAIN_AREA_FRACTION_QUANTILE)
    slice_summary_df["low_pred_brain_area"] = slice_summary_df["brain_area_fraction_pred"] <= pred_cutoff

print(f"Loaded checkpoint tables for {slice_summary_df['case'].nunique()} cases and {len(slice_summary_df)} slices")
print(slice_summary_df[["case", "slice_idx_original", "gt_brain_pixels", "pred_brain_pixels", "brain_dice", "brain_area_fraction_gt", "brain_area_fraction_pred"]].head())

selected_slices_df = build_selected_slices(slice_summary_df)
exported_rows = []
case_lookup = {case_dir.name: case_dir for case_dir in case_dirs}
case_cache = {}

def get_case_context(case_name):
    if case_name not in case_cache:
        case_dir = case_lookup[case_name]
        _, volume = load_nifti_volume(case_dir / "Seq No.nii")
        brain_masks = load_union_mask_dict(case_dir / "cerebro", volume.shape)
        case_cache[case_name] = {
            "volume": volume,
            "brain_masks": brain_masks,
        }
    return case_cache[case_name]

for _, selected_row in selected_slices_df.iterrows():
    case_name = selected_row["case"]
    slice_idx = int(selected_row["slice_idx_original"])
    context = get_case_context(case_name)

    slice_img = context["volume"][:, :, slice_idx]
    gt_mask = resize_mask(
        context["brain_masks"].get(slice_idx, np.zeros_like(slice_img, dtype=np.uint8)),
        target_size=IMG_TARGET,
    )

    prediction = predict_brain_slice(slice_img, model_brain=model_brain)
    xai_bundle = compute_xai_bundle(
        model=model_brain,
        input_tensor=prediction["input_tensor"],
        roi_mask=prediction["roi_mask"],
        target_layer_name=resolved_target_layer,
        baseline_image=prediction["baseline_image"],
        ig_steps=IG_STEPS,
        curve_steps=CURVE_STEPS,
        occlusion_patch=OCCLUSION_PATCH,
        occlusion_stride=OCCLUSION_STRIDE,
        include_occlusion=True,
        faithfulness_methods=None,
    )

    total_pixels = int(IMG_TARGET * IMG_TARGET)
    gt_brain_pixels = int(gt_mask.sum())
    pred_brain_pixels = int(prediction["brain_pred_mask"].sum())
    brain_dice = float(dice_numpy(gt_mask, prediction["brain_pred_mask"]))
    brain_area_ratio_pred_gt = float(pred_brain_pixels / (gt_brain_pixels + 1e-8))

    artifact = {
        "image": prediction["resized_img"],
        "gt_mask": gt_mask,
        "pred_mask": prediction["brain_pred_mask"],
        "gradcam": xai_bundle["attribution_maps"]["gradcam"],
        "gradcampp": xai_bundle["attribution_maps"]["gradcampp"],
        "ig": xai_bundle["attribution_maps"]["ig"],
        "occlusion": xai_bundle["occlusion_map"],
        "faithfulness_by_method": xai_bundle["faithfulness_by_method"],
        "meta": {
            "case": case_name,
            "slice_idx_original": slice_idx,
            "gt_brain_pixels": gt_brain_pixels,
            "pred_brain_pixels": pred_brain_pixels,
            "brain_area_ratio_pred_gt": brain_area_ratio_pred_gt,
            "brain_area_fraction_gt": float(gt_brain_pixels / total_pixels),
            "brain_area_fraction_pred": float(pred_brain_pixels / total_pixels),
            "brain_dice": brain_dice,
            "target_mask_strategy_used": prediction["roi_strategy"],
            "has_explainable_target": bool(prediction["has_explainable_target"]),
        },
    }

    output_path = FIGURES_DIR / selected_row["selection_group"] / make_export_filename(case_name, slice_idx)
    render_slice_montage(artifact, output_path)
    exported_rows.append({
        "slice_uid": selected_row["slice_uid"],
        "case": case_name,
        "slice_idx_original": slice_idx,
        "selection_group": selected_row["selection_group"],
        "selection_reason": selected_row["selection_reason"],
        "figure_path": str(output_path),
    })
    pd.DataFrame(exported_rows).to_csv(CSV_DIR / "selected_slices.csv", index=False)

    del prediction
    del xai_bundle
    del artifact
    gc.collect()

selected_slices_df = pd.DataFrame(exported_rows)

if not selected_slices_df.empty:
    group_map = selected_slices_df.groupby("slice_uid")["selection_group"].agg(lambda values: "|".join(sorted(set(values))))
    reason_map = selected_slices_df.groupby("slice_uid")["selection_reason"].agg(lambda values: "|".join(values))
    figure_map = selected_slices_df.groupby("slice_uid")["figure_path"].agg(lambda values: "|".join(values))

    slice_summary_df.loc[slice_summary_df["slice_uid"].isin(group_map.index), "selected_for_export"] = True
    slice_summary_df["selected_export_groups"] = slice_summary_df["slice_uid"].map(group_map).fillna("")
    slice_summary_df["selection_reasons"] = slice_summary_df["slice_uid"].map(reason_map).fillna("")
    slice_summary_df["figure_paths"] = slice_summary_df["slice_uid"].map(figure_map).fillna("")
    method_metrics_df["figure_paths"] = method_metrics_df["slice_uid"].map(figure_map).fillna("")

print(f"Exported {len(selected_slices_df)} montage files")
selected_slices_df.head()


## 12. Exportación final e interpretación

Esta última fase consolida los ficheros CSV y resume el número de casos, cortes y ejemplos exportados. Las tablas resultantes permiten analizar por separado rendimiento de segmentación, selección de cortes, métricas por método y curvas de perturbación.

Notas de interpretación para la memoria:

- La explicación se calcula sobre la máscara cerebral predicha, porque se busca inspeccionar la predicción producida por el modelo.
- La máscara manual se utiliza para evaluar concordancia mediante Dice, IoU y medidas de área, no para forzar la explicación.
- Los cortes sin región cerebral predicha no tienen un objetivo de primer plano explicable; por ello sus métricas de fidelidad se exportan como `NaN` y no deben promediarse como fallos de XAI.
- Un mapa XAI visualmente plausible no demuestra que el modelo razone de forma causal; solo sugiere que la señal usada por el modelo puede estar espacialmente alineada con la anatomía segmentada.
- Grad-CAM++ se implementa con una aproximación conservadora orientada a memoria, documentada porque la formulación exacta de orden superior puede ser costosa en modelos de segmentación.


In [ ]:
# ============================================================
# Final CSV exports
# ============================================================

slice_summary_path = CSV_DIR / "slice_summary.csv"
method_metrics_path = CSV_DIR / "method_metrics_long.csv"
curve_points_path = CSV_DIR / "curve_points_long.csv"
curve_points_checkpoint_path = CSV_DIR / "curve_points_checkpoint.csv"
group_summary_path = CSV_DIR / "group_summary.csv"
selected_slices_path = CSV_DIR / "selected_slices.csv"

if "slice_summary_df" not in globals() or "method_metrics_df" not in globals() or "curve_points_df" not in globals():
    slice_summary_df, method_metrics_df, curve_points_df = load_checkpoint_tables()

if "low_gt_brain_area" not in slice_summary_df.columns:
    gt_cutoff = slice_summary_df["brain_area_fraction_gt"].quantile(LOW_BRAIN_AREA_FRACTION_QUANTILE)
    slice_summary_df["low_gt_brain_area"] = slice_summary_df["brain_area_fraction_gt"] <= gt_cutoff
if "low_pred_brain_area" not in slice_summary_df.columns:
    pred_cutoff = slice_summary_df["brain_area_fraction_pred"].quantile(SMALL_PRED_BRAIN_AREA_FRACTION_QUANTILE)
    slice_summary_df["low_pred_brain_area"] = slice_summary_df["brain_area_fraction_pred"] <= pred_cutoff

if "selected_slices_df" not in globals():
    if selected_slices_path.exists():
        selected_slices_df = pd.read_csv(selected_slices_path)
    else:
        selected_slices_df = pd.DataFrame(columns=["slice_uid", "case", "slice_idx_original", "selection_group", "selection_reason", "figure_path"])

group_summary_df = build_group_summary(method_metrics_df, slice_summary_df)

slice_summary_df.to_csv(slice_summary_path, index=False)
method_metrics_df.to_csv(method_metrics_path, index=False)
if curve_points_df.empty and curve_points_checkpoint_path.exists():
    curve_points_df = pd.read_csv(curve_points_checkpoint_path)
curve_points_df.to_csv(curve_points_path, index=False)
group_summary_df.to_csv(group_summary_path, index=False)

if selected_slices_df.empty:
    pd.DataFrame(columns=["slice_uid", "case", "slice_idx_original", "selection_group", "selection_reason", "figure_path"]).to_csv(selected_slices_path, index=False)
else:
    selected_slices_df.to_csv(selected_slices_path, index=False)

print("Saved outputs:")
print(f"  - {slice_summary_path}")
print(f"  - {method_metrics_path}")
print(f"  - {curve_points_path}")
print(f"  - {group_summary_path}")
print(f"  - {selected_slices_path}")

print("\nSummary counts")
print(f"  Cases: {slice_summary_df['case'].nunique()}")
print(f"  Slices: {len(slice_summary_df)}")
print(f"  Edge first/last 3 slices: {int(slice_summary_df['is_edge_first_last3'].sum())}")
print(f"  Interior slices: {int(slice_summary_df['is_interior'].sum())}")
print(f"  Slices included in quantitative XAI metrics: {int(slice_summary_df['has_explainable_target'].sum())}")
print(f"  Low GT brain area slices: {int(slice_summary_df['low_gt_brain_area'].sum())}")
print(f"  Low predicted brain area slices: {int(slice_summary_df['low_pred_brain_area'].sum())}")
print(f"  Exported montages: {len(selected_slices_df)}")
